In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load the telemetry data
metric_df = pd.read_csv('metric.csv')
log_df = pd.read_csv('log.csv')
trace_df = pd.read_csv('trace.csv')

# Set timezone to UTC+8
utc_tz = pytz.timezone('Asia/Shanghai')

# Convert timestamp to datetime
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize(utc_tz)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize(utc_tz)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize(utc_tz)

# Define possible root cause components
components = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Filter metrics for relevant components
metric_filtered = metric_df[metric_df['cmdb_id'].isin(components)]
log_filtered = log_df[log_df['cmdb_id'].isin(components)]
trace_filtered = trace_df[trace_df['cmdb_id'].isin(components)]

# Calculate global thresholds (P95) for each component-KPI in metrics
metric_thresholds = metric_filtered.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index()
metric_thresholds.columns = ['cmdb_id', 'kpi_name', 'p95_value']

# Calculate global thresholds (P95) for each component-log in logs
log_thresholds = log_filtered.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.95).reset_index()
log_thresholds.columns = ['cmdb_id', 'log_name', 'p95_value']

# Calculate global thresholds (P95) for each component-trace in traces
trace_thresholds = trace_filtered.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).reset_index()
trace_thresholds.columns = ['cmdb_id', 'trace_name', 'p95_value']

# Combine all thresholds into a single DataFrame
combined_thresholds = pd.concat([
    metric_thresholds.assign(type='metric'),
    log_thresholds.assign(type='log'),
    trace_thresholds.assign(type='trace')
], ignore_index=True)

# Display the combined thresholds summary
combined_thresholds_summary = combined_thresholds.groupby(['cmdb_id', 'type']).agg(
    count=('p95_value', 'size'),
    earliest_timestamp=('p95_value', 'min'),
    max_value=('p95_value', 'max'),
    min_value=('p95_value', 'min')
).reset_index()

combined_thresholds_summary.head(20)

```
Out[1]:
```
The global thresholds (P95 values) for each component across different telemetry types (logs, metrics, and traces) have been calculated. The summary shows the count of records, earliest timestamps, maximum values, and minimum values for each component. All components have a minimum value of 0.0, indicating no negative values in the data. The maximum values vary significantly across components and telemetry types, with the highest being approximately 4.53e+07 for metrics in the currency service.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    type  count  earliest_timestamp     max_value  min_value
0         checkoutservice     log      2                 0.0  1.236000e+02        0.0
1         checkoutservice  metric      7                 0.0  1.802199e+07        0.0
2         checkoutservice   trace     32                 0.0  3.376000e+02        0.0
3         currencyservice     log      2                 0.0  2.457800e+03        0.0
4         currencyservice  metric      7                 0.0  4.535624e+07        0.0
5         currencyservice   trace     12                 0.0  2.351400e+03        0.0
6            emailservice     log      2                 0.0  4.120000e+01        0.0
7            emailservice  metric      7                 0.0  4.361923e+07        0.0
8            emailservice   trace      8                 0.0  4.160000e+01        0.0
9   productcatalogservice  metric      7                 0.0  1.058359e+07        0.0
10  productcatalogservice   trace     16                 0.0  3.894900e+03        0.0
11  recommendationservice     log      2                 0.0  6.052000e+02        0.0
12  recommendationservice  metric      7                 0.0  4.527991e+07        0.0
13  recommendationservice   trace     20                 0.0  6.058000e+02        0.0```
```

In [ ]:
# In[2]:

# Define the incident window
start_time = pd.Timestamp('2024-02-21 00:14:21').tz_localize(utc_tz)
end_time = pd.Timestamp('2024-02-21 00:44:21').tz_localize(utc_tz)

# Filter the data within the incident window
metric_filtered_window = metric_df[(metric_df['timestamp'] >= start_time) & (metric_df['timestamp'] <= end_time)]
log_filtered_window = log_df[(log_df['timestamp'] >= start_time) & (log_df['timestamp'] <= end_time)]
trace_filtered_window = trace_df[(trace_df['timestamp'] >= start_time) & (trace_df['timestamp'] <= end_time)]

# Display the shape of the filtered DataFrames
metric_filtered_window_shape = metric_filtered_window.shape
log_filtered_window_shape = log_filtered_window.shape
trace_filtered_window_shape = trace_filtered_window.shape

metric_filtered_window_shape, log_filtered_window_shape, trace_filtered_window_shape

```
Out[2]:
```
The filtering of the telemetry data within the incident window resulted in 1,154 records from the metrics, 284 records from the logs, and 1,920 records from the traces.

The original code execution output of IPython Kernel is also provided below for reference:

((1154, 4), (284, 4), (1920, 4))```
```

In [ ]:
# In[3]:

# Merge the filtered metrics data with the global thresholds to identify anomalies
anomalies = metric_filtered_window.merge(combined_thresholds[combined_thresholds['type'] == 'metric'],
                                         left_on=['cmdb_id', 'kpi_name'],
                                         right_on=['cmdb_id', 'kpi_name'],
                                         suffixes=('', '_threshold'))

# Identify anomalies where the value exceeds the P95 threshold
anomalies_detected = anomalies[anomalies['value'] > anomalies['p95_value']]

# Count the number of anomalies for each component-KPI
anomaly_counts = anomalies_detected.groupby(['cmdb_id', 'kpi_name']).size().reset_index(name='anomaly_count')

# Display the summary of anomaly counts
anomaly_summary = anomaly_counts.sort_values(by='anomaly_count', ascending=False).head(20)

anomaly_summary

```
Out[3]:
```
Anomaly detection on the filtered metrics data revealed that several KPIs exceeded the global thresholds. Each of the following components had one anomaly detected for each KPI: 

- **Checkout Service**: CPU, latency-50, latency-90, memory, socket, and workload.
- **Currency Service**: CPU, latency-50, latency-90, memory, and workload.
- **Email Service**: CPU, latency-50, latency-90, memory, and workload.
- **Product Catalog Service**: CPU, latency-50, latency-90, and memory.

In total, each component showed one anomaly for multiple KPIs, indicating potential issues across these metrics.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  anomaly_count
0         checkoutservice         cpu              1
1         checkoutservice  latency-50              1
2         checkoutservice  latency-90              1
3         checkoutservice         mem              1
4         checkoutservice      socket              1
5         checkoutservice    workload              1
6         currencyservice         cpu              1
7         currencyservice  latency-50              1
8         currencyservice  latency-90              1
9         currencyservice         mem              1
10        currencyservice    workload              1
11           emailservice         cpu              1
12           emailservice  latency-50              1
13           emailservice  latency-90              1
14           emailservice         mem              1
15           emailservice    workload              1
16  productcatalogservice         cpu              1
17  productcatalogservice  latency-50              1
18  productcatalogservice  latency-90              1
19  productcatalogservice         mem              1```
```

In [ ]:
# In[4]:

# Filter the traces for the components identified with anomalies
faulty_components = anomaly_summary['cmdb_id'].unique()
trace_filtered_faults = trace_filtered_window[trace_filtered_window['cmdb_id'].isin(faulty_components)]

# Count the number of faults and extract the earliest timestamps for each component
trace_fault_summary = trace_filtered_faults.groupby('cmdb_id').agg(
    fault_count=('value', 'size'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Display the summary of trace faults
trace_fault_summary.head(20)

```
Out[4]:
```
The analysis of the trace data identified the following faulty components along with their fault counts and earliest timestamps:

- **Checkout Service**: 412 faults, earliest timestamp at 2024-02-21 00:17:00 UTC+8.
- **Currency Service**: 156 faults, earliest timestamp at 2024-02-21 00:17:00 UTC+8.
- **Email Service**: 100 faults, earliest timestamp at 2024-02-21 00:17:00 UTC+8.
- **Product Catalog Service**: 192 faults, earliest timestamp at 2024-02-21 00:17:00 UTC+8.

All components experienced faults starting at the same earliest timestamp.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  fault_count        earliest_timestamp
0        checkoutservice          412 2024-02-21 00:17:00+08:00
1        currencyservice          156 2024-02-21 00:17:00+08:00
2           emailservice          100 2024-02-21 00:17:00+08:00
3  productcatalogservice          192 2024-02-21 00:17:00+08:00```
```

In [ ]:
# In[5]:

# Since the log.csv does not have a 'message' column, we will adjust the code to focus on the available columns.
# We will count the log entries and extract the earliest timestamps for the Checkout Service.

# Count the number of log entries and extract the earliest timestamp
log_summary = checkout_logs.groupby('log_name').agg(
    log_count=('value', 'size'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Display the summary of log entries for the Checkout Service
log_summary.head(20)

```
Out[5]:
```
The analysis of the logs for the Checkout Service revealed the following:

- **Error Count Log**: 15 entries, with the earliest timestamp at 2024-02-21 00:17:00 UTC+8.
- **Row Count Log**: 15 entries, with the earliest timestamp at 2024-02-21 00:17:00 UTC+8.

Both log types indicate activity starting at the same earliest timestamp, which may be relevant to the failure reason.

The original code execution output of IPython Kernel is also provided below for reference:

log_name  log_count        earliest_timestamp
0  log.error_count         15 2024-02-21 00:17:00+08:00
1    log.row_count         15 2024-02-21 00:17:00+08:00```
```